# Демонстрация 3: маленький `spa-eng` seq2seq с attention

Перед запуском полезно открыть dataset card [English-Spanish pairs](./cards/03_spa_eng.md).

Fast-mode идея:
- взять короткие англо-испанские пары;
- построить компактный `GRU seq2seq + attention`;
- посмотреть не только на token-level метрику, но и на одну heatmap.


In [1]:
import pathlib
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(17)
np.random.seed(17)
random.seed(17)


2026-03-24 11:00:48.751531: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 11:00:48.751944: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 11:00:48.753881: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 11:00:48.759416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774339248.769567  194600 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774339248.77

## Выбор runtime

Здесь вы выбираете, где и на чём запускать notebook.

Что обычно выбирать:
- `auto` — лучший вариант по умолчанию. Если TensorFlow видит GPU, будет выбран GPU. Если GPU нет, notebook спокойно останется на CPU.
- `local-cpu` — локальный запуск только на CPU, даже если видеокарта есть.
- `local-gpu` — локальный запуск с обязательным GPU. Если GPU не настроен, notebook специально остановится с понятной ошибкой.
- `colab-cpu` / `colab-gpu` — запуск в Google Colab.
- `kaggle-cpu` / `kaggle-gpu` — запуск в Kaggle Notebooks.

Что важно:
- после изменения `RUNTIME_MODE` используйте `Restart & Run All`;
- `COURSE_REPO_HTTPS_URL` нужен только для Colab/Kaggle, если репозиторий ещё не клонирован в runtime;
- пока в ячейке стоит placeholder-URL, cloud auto-bootstrap не сможет сам скачать курс;
- guide `05` отвечает на вопрос, где и как запускать notebook;
- guide `06` нужен, если вы хотите именно локальный GPU и не уверены в версиях `TensorFlow` / `CUDA`;
- локальный GPU-path курса: `Linux + NVIDIA` или `Windows -> WSL2 + Ubuntu`;
- если `local-gpu` упирается в локальные CUDA/PTX ошибки, это обычно уже проблема GPU-стека, а не notebook. В таком случае спокойно переключайтесь на `local-cpu`, `colab-gpu` или `kaggle-gpu`.

Подробные guides:
- `themes/00-Foundations/guides/05_local_tensorflow_gpu_notebooks.md`
- `themes/00-Foundations/guides/06_tensorflow_cuda_version_selection.md`


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/00-Foundations/requirements.txt"


def _detect_notebook_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


In [2]:
zip_path = tf.keras.utils.get_file(
    fname='spa-eng.zip',
    origin='https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip',
    extract=True,
)
data_path = pathlib.Path(zip_path).with_suffix('') / 'spa-eng' / 'spa.txt'

def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'([?.!,¿])', r' \1 ', text)
    text = re.sub(r'[^a-záéíóúüñ¿?.!,]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

pairs = []
for line in data_path.read_text(encoding='utf-8').splitlines():
    if '	' not in line:
        continue
    english, spanish = line.split('	')[:2]
    source = normalize_text(spanish)
    target = normalize_text(english)
    if len(source.split()) <= 6 and len(target.split()) <= 6:
        pairs.append((source, '[start] ' + target + ' [end]'))

random.shuffle(pairs)
pairs = pairs[:1800]

source_texts = [src for src, tgt in pairs]
target_texts = [tgt for src, tgt in pairs]

src_train, src_val, tgt_train, tgt_val = train_test_split(
    source_texts,
    target_texts,
    test_size=0.2,
    random_state=17,
)

print('pairs in fast-mode subset:', len(pairs))
print('example source:', src_train[0])
print('example target:', tgt_train[0])


      0/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

  16384/2638744 ━━━━━━━━━━━━━━━━━━━━ 10s 4us/step

  49152/2638744 ━━━━━━━━━━━━━━━━━━━━ 8s 3us/step 

  81920/2638744 ━━━━━━━━━━━━━━━━━━━━ 7s 3us/step

 131072/2638744 ━━━━━━━━━━━━━━━━━━━━ 5s 2us/step

...

2433024/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

2514944/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

2596864/2638744 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step


pairs in fast-mode subset: 1800
example source: ella nunca piensa en él .
example target: [start] she never thinks about him . [end]


In [3]:
source_seq_len = 12
target_seq_len = 14
vocab_limit = 2000

source_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_limit,
    output_mode='int',
    output_sequence_length=source_seq_len,
)
target_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_limit,
    output_mode='int',
    output_sequence_length=target_seq_len,
)

source_vectorizer.adapt(np.array(src_train))
target_vectorizer.adapt(np.array(tgt_train))

encoder_train = source_vectorizer(np.array(src_train)).numpy()
encoder_val = source_vectorizer(np.array(src_val)).numpy()

target_train_full = target_vectorizer(np.array(tgt_train)).numpy()
target_val_full = target_vectorizer(np.array(tgt_val)).numpy()

decoder_input_train = target_train_full[:, :-1]
decoder_target_train = target_train_full[:, 1:]
decoder_input_val = target_val_full[:, :-1]
decoder_target_val = target_val_full[:, 1:]

sample_weight_train = (decoder_target_train != 0).astype('float32')
sample_weight_val = (decoder_target_val != 0).astype('float32')

print('encoder_train shape      :', encoder_train.shape)
print('decoder_input_train shape:', decoder_input_train.shape)
print('decoder_target_train shape:', decoder_target_train.shape)


encoder_train shape      : (1440, 12)
decoder_input_train shape: (1440, 13)
decoder_target_train shape: (1440, 13)


E0000 00:00:1774339253.861749  194600 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774339253.864369  194600 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
source_vocab_size = len(source_vectorizer.get_vocabulary())
target_vocab_size = len(target_vectorizer.get_vocabulary())
embed_dim = 64
hidden_dim = 64

encoder_tokens = tf.keras.Input(shape=(source_seq_len,), name='encoder_tokens')
decoder_tokens = tf.keras.Input(shape=(target_seq_len - 1,), name='decoder_tokens')

encoder_embed = tf.keras.layers.Embedding(source_vocab_size, embed_dim, mask_zero=True, name='enc_embedding')(encoder_tokens)
encoder_outputs, encoder_state = tf.keras.layers.GRU(
    hidden_dim, return_sequences=True, return_state=True, name='enc_gru'
)(encoder_embed)

decoder_embed = tf.keras.layers.Embedding(target_vocab_size, embed_dim, mask_zero=True, name='dec_embedding')(decoder_tokens)
decoder_outputs = tf.keras.layers.GRU(
    hidden_dim, return_sequences=True, name='dec_gru'
)(decoder_embed, initial_state=encoder_state)

attention_layer = tf.keras.layers.Attention(name='cross_attention')
context, attention_scores = attention_layer(
    [decoder_outputs, encoder_outputs],
    return_attention_scores=True,
)

combined = tf.keras.layers.Concatenate(name='decoder_context_concat')([decoder_outputs, context])
logits = tf.keras.layers.Dense(target_vocab_size, activation='softmax', name='token_probs')(combined)

training_model = tf.keras.Model([encoder_tokens, decoder_tokens], logits)
analysis_model = tf.keras.Model([encoder_tokens, decoder_tokens], [logits, attention_scores])

training_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='token_accuracy')],
)

history = training_model.fit(
    [encoder_train, decoder_input_train],
    decoder_target_train,
    sample_weight=sample_weight_train,
    validation_data=([
        encoder_val,
        decoder_input_val,
    ], decoder_target_val, sample_weight_val),
    epochs=4,
    batch_size=64,
    verbose=0,
)


2026-03-24 11:00:55.502314: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['token_accuracy'], marker='o', label='train')
axes[0].plot(history.history['val_token_accuracy'], marker='o', label='val')
axes[0].set_title('spa-eng fast-mode: token accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], marker='o', label='train')
axes[1].plot(history.history['val_loss'], marker='o', label='val')
axes[1].set_title('spa-eng fast-mode: loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


/tmp/ipykernel_194600/2951814832.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
source_vocab = np.array(source_vectorizer.get_vocabulary())
target_vocab = np.array(target_vectorizer.get_vocabulary())

row = 0
sample_encoder = encoder_val[row : row + 1]
sample_decoder = decoder_input_val[row : row + 1]
probs, attention_scores = analysis_model.predict([sample_encoder, sample_decoder], verbose=0)

pred_ids = probs.argmax(axis=-1)[0]
gold_ids = decoder_target_val[row]

def decode(ids, vocab, stop_token='[end]'):
    tokens = []
    for idx in ids:
        token = vocab[idx]
        if token == '' or token == '[start]':
            continue
        if token == stop_token:
            break
        tokens.append(token)
    return tokens

source_tokens = [token for token in src_val[row].split()]
gold_tokens = decode(gold_ids, target_vocab)
pred_tokens = decode(pred_ids, target_vocab)

print('source text:', src_val[row])
print('gold text  :', ' '.join(gold_tokens))
print('pred text  :', ' '.join(pred_tokens))


source text: yo hago todo el trabajo .
gold text  : i do all the work end
pred text  : i end end end end end end end end end end end end


In [7]:
heatmap = attention_scores[0]
row_labels = gold_tokens if gold_tokens else [f'step_{i}' for i in range(heatmap.shape[0])]
col_labels = source_tokens if source_tokens else [f'src_{i}' for i in range(heatmap.shape[1])]

heatmap = heatmap[: len(row_labels), : len(col_labels)]

plt.figure(figsize=(8, 5))
plt.imshow(heatmap, cmap='magma', aspect='auto')
plt.xticks(range(len(col_labels)), col_labels, rotation=45, ha='right')
plt.yticks(range(len(row_labels)), row_labels)
plt.xlabel('Spanish source tokens')
plt.ylabel('English target steps')
plt.title('Attention heatmap on one validation example')
plt.colorbar(label='attention weight')
plt.tight_layout()
plt.show()


/tmp/ipykernel_194600/548997373.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Что заметить

- Здесь уже появляется полный `seq2seq` интерфейс: `encoder_input`, `decoder_input`, `decoder_target`.
- `token_accuracy` полезна, но сама по себе не гарантирует хорошую последовательность.
- Даже маленький fast-mode run уже даёт heatmap, по которой можно обсуждать alignment между source и target.
